# `Dynamic AI Knowledge Assistant`

In [1]:
# ===== Import Libraries =========

from pathlib import Path
import re
import unicodedata
import hashlib
import pandas as pd
import numpy as np
from collections import Counter

In [2]:
# ======== Project Paths =========

# Project root
PROJECT_ROOT = Path.cwd().parent
# Raw Wikipedia documents
RAW_DIR = PROJECT_ROOT / "data" / "raw" / "wikipedia"
# Processed cleaned documents
CLEANED_DIR = PROJECT_ROOT / "data" / "processed" / "cleaned"
# Create processed directory if it does not exist
CLEANED_DIR.mkdir(parents=True, exist_ok=True)
print("Project Root :", PROJECT_ROOT)
print("Raw Directory:", RAW_DIR)
print("Cleaned Directory:", CLEANED_DIR)

Project Root : c:\Users\RAaM\OneDrive\Desktop\Dynamic AI Knowledge Assistant
Raw Directory: c:\Users\RAaM\OneDrive\Desktop\Dynamic AI Knowledge Assistant\data\raw\wikipedia
Cleaned Directory: c:\Users\RAaM\OneDrive\Desktop\Dynamic AI Knowledge Assistant\data\processed\cleaned


In [3]:
# ========= Verify the Raw Files =========

raw_files = sorted(RAW_DIR.glob("*.txt"))

print("Total TXT files:", len(raw_files))

if len(raw_files) == 0:
    raise FileNotFoundError(
        f"No TXT files found in: {RAW_DIR}"
    )

for file in raw_files[:20]:
    print(file.name)

Total TXT files: 128
1.58-bit_large_language_model.txt
A.I._Artificial_Intelligence.txt
Active_learning_(machine_learning).txt
Agent_harness.txt
AI_art.txt
AI_boom.txt
AI_literacy.txt
AI_slop.txt
AlexNet.txt
Artificial_general_intelligence.txt
Artificial_intelligence.txt
Artificial_Intelligence_Act.txt
Artificial_intelligence_and_copyright.txt
Attention_(machine_learning).txt
Automated_machine_learning.txt
Automatic1111.txt
Bob_van_Luijt.txt
ChatGPT.txt
Chroma.txt
ChromaDB.txt


In [4]:
# ======== Load Raw Documents ========

documents = []
for file_path in raw_files:
    text = file_path.read_text(
        encoding="utf-8",
        errors="replace"
    )
    documents.append({
        "file_name": file_path.name,
        "file_path": str(file_path),
        "text": text
    })

raw_df = pd.DataFrame(documents)
print("Documents loaded:", len(raw_df))
print("Total characters:", raw_df["text"].str.len().sum())

Documents loaded: 128
Total characters: 2583893


In [5]:
raw_df["character_count"] = raw_df["text"].str.len()

raw_df["word_count"] = raw_df["text"].apply(
    lambda x: len(re.findall(r"\S+", x))
)
raw_df["line_count"] = raw_df["text"].str.count("\n") + 1
raw_df["blank_line_count"] = raw_df["text"].apply(
    lambda x: sum(1 for line in x.splitlines() if not line.strip())
)
raw_df[
    [
        "file_name",
        "character_count",
        "word_count",
        "line_count",
        "blank_line_count"
    ]
].head()

,file_name,character_count,word_count,line_count,blank_line_count
0,1.58-bit_large_language_model.txt,4395,609,50,21
1,A.I._Artificial_Intelligence.txt,26064,4226,145,50
2,Active_learning_(machine_learning).txt,10713,1545,111,54
3,Agent_harness.txt,5017,751,31,10
4,AI_art.txt,22334,3305,101,41


In [6]:
# ========== Document Length Statistics ========

length_statistics = raw_df[
    ["character_count", "word_count", "line_count", "blank_line_count"]
].describe()

length_statistics

,character_count,word_count,line_count,blank_line_count
count,128.000000,128.000000,128.000000,128.000000
mean,20186.664062,2795.906250,306.632812,155.062500
std,23324.972519,3208.514517,637.185802,372.134696
min,414.000000,61.000000,10.000000,4.000000
25%,4221.250000,602.000000,42.750000,14.000000
50%,12635.500000,1633.000000,97.000000,37.500000
75%,30147.500000,4068.500000,268.250000,112.500000
max,138132.000000,20279.000000,5682.000000,3258.000000


In [7]:
# ====== Identify Empty  Documents =========

empty_documents = raw_df[
    raw_df["text"].str.strip() == ""
]

print("Empty documents:", len(empty_documents))

if len(empty_documents) > 0:
    display(
        empty_documents[
            ["file_name", "character_count", "word_count"]
        ]
    )


Empty documents: 0


In [8]:
# ======== Identify Short Documents ========

SHORT_DOCUMENT_THRESHOLD = 100
short_documents = raw_df[
    raw_df["word_count"] < SHORT_DOCUMENT_THRESHOLD
].copy()
print(
    f"Documents with fewer than "
    f"{SHORT_DOCUMENT_THRESHOLD} words:",
    len(short_documents)
)
display(
    short_documents[
        [
            "file_name",
            "character_count",
            "word_count",
            "line_count"
        ]
    ].sort_values("word_count")
)

for _, row in short_documents.iterrows():
    print("=" * 80)
    print("FILE:", row["file_name"])
    print("WORDS:", row["word_count"])
    print("=" * 80)
    print(row["text"][:2000])
    print()

Documents with fewer than 100 words: 1


,file_name,character_count,word_count,line_count
19,ChromaDB.txt,414,61,10


FILE: ChromaDB.txt
WORDS: 61
Chroma or ChromaDB is open-source data infrastructure tailored to applications with large language models.
Its headquarters are in San Francisco. In April 2023, it raised 18 million US dollars as seed funding.
ChromaDB has been used in academic studies on artificial intelligence, particularly as part of the tech stack for retrieval-augmented generation.


== References ==


== External links ==
Official website



In [9]:
# ======= Check White Space Problem =======

def whitespace_metrics(text):
    return {
        "leading_whitespace": len(text) - len(text.lstrip()),
        "trailing_whitespace": len(text) - len(text.rstrip()),
        "double_space_count": len(re.findall(r" {2,}", text)),
        "tab_count": text.count("\t"),
        "blank_line_count": sum(
            1 for line in text.splitlines()
            if not line.strip()
        )
    }


whitespace_data = raw_df["text"].apply(whitespace_metrics)

whitespace_df = pd.DataFrame(
    whitespace_data.tolist()
)

raw_df = pd.concat(
    [raw_df, whitespace_df],
    axis=1
)

display(
    raw_df[
        [
            "file_name",
            "leading_whitespace",
            "trailing_whitespace",
            "double_space_count",
            "tab_count",
            "blank_line_count"
        ]
    ].head(20)
)

,file_name,leading_whitespace,trailing_whitespace,double_space_count,tab_count,blank_line_count,blank_line_count
0,1.58-bit_large_language_model.txt,0,0,18,0,21,21
1,A.I._Artificial_Intelligence.txt,0,0,1,0,50,50
2,Active_learning_(machine_learning).txt,0,0,58,0,54,54
3,Agent_harness.txt,0,0,0,0,10,10
4,AI_art.txt,0,0,1,0,41,41
5,AI_boom.txt,0,0,2,0,61,61
6,AI_literacy.txt,0,0,2,0,30,30
7,AI_slop.txt,0,0,1,0,51,51
8,AlexNet.txt,0,0,0,0,20,20
9,Artificial_general_intelligence.txt,0,0,1,0,88,88


In [10]:
whitespace_issue_mask = (
    (raw_df["leading_whitespace"] > 0) |
    (raw_df["trailing_whitespace"] > 0) |
    (raw_df["double_space_count"] > 0) |
    (raw_df["tab_count"] > 0)
)

print(
    "Documents with potential whitespace issues:",
    whitespace_issue_mask.sum()
)

Documents with potential whitespace issues: 78


In [11]:
# ======== Inspect unicode Characteristics =========

def unicode_category_counts(text):
    counter = Counter()

    for char in text:
        counter[unicodedata.category(char)] += 1

    return counter


unicode_category_data = raw_df["text"].apply(
    unicode_category_counts
)

unicode_category_totals = Counter()

for counter in unicode_category_data:
    unicode_category_totals.update(counter)

unicode_category_df = pd.DataFrame(
    unicode_category_totals.items(),
    columns=["unicode_category", "count"]
).sort_values(
    "count",
    ascending=False
)

unicode_category_df

,unicode_category,count
5,Ll,1810239
1,Zs,544268
0,Lu,71602
3,Po,51847
9,Cc,39121
2,Nd,26237
8,Sm,14156
6,Ps,8393
7,Pe,8393
4,Pd,7902


In [12]:
# ======== Detect Control Characters =====

def find_control_characters(text):
    controls = []

    for char in text:
        category = unicodedata.category(char)

        if category == "Cc" and char not in "\n\r\t":
            controls.append(char)

    return controls


raw_df["control_characters"] = raw_df["text"].apply(
    find_control_characters
)

raw_df["control_character_count"] = (
    raw_df["control_characters"].str.len()
)

control_issue_df = raw_df[
    raw_df["control_character_count"] > 0
][
    [
        "file_name",
        "control_character_count",
        "control_characters"
    ]
]

print(
    "Documents containing non-standard control characters:",
    len(control_issue_df)
)

display(control_issue_df)

Documents containing non-standard control characters: 0


,file_name,control_character_count,control_characters


In [13]:
# ======== Inspect Non - ASCII Contents ========

def non_ascii_count(text):
    return sum(1 for char in text if ord(char) > 127)


raw_df["non_ascii_count"] = raw_df["text"].apply(
    non_ascii_count
)

raw_df["non_ascii_percentage"] = (
    raw_df["non_ascii_count"]
    / raw_df["character_count"].replace(0, np.nan)
) * 100

display(
    raw_df[
        [
            "file_name",
            "character_count",
            "non_ascii_count",
            "non_ascii_percentage"
        ]
    ].sort_values(
        "non_ascii_percentage",
        ascending=False
    ).head(20)
)

,file_name,character_count,non_ascii_count,non_ascii_percentage
89,Outline_of_natural_language_processing.txt,50041,349,0.697428
69,List_of_data_science_software.txt,5292,31,0.585790
59,Interim_Measures_for_the_Management_of_Generat...,5794,30,0.517777
48,Generative_AI_pornography.txt,1293,6,0.464037
68,List_of_computer_science_conferences.txt,10853,42,0.386990
105,Residual_neural_network.txt,25797,83,0.321743
125,Triangulation_(computer_vision).txt,31713,88,0.277489
118,Timeline_of_artificial_intelligence.txt,7231,18,0.248928
103,Reinforcement_learning.txt,52531,121,0.230340
45,Fundamental_matrix_(computer_vision).txt,13466,30,0.222783


In [14]:
# ========== Detect Duplicate Content ==========

def content_hash(text):
    normalized_for_hash = text.strip()

    return hashlib.sha256(
        normalized_for_hash.encode("utf-8")
    ).hexdigest()


raw_df["content_hash"] = raw_df["text"].apply(
    content_hash
)

duplicate_hash_groups = (
    raw_df.groupby("content_hash")
    .size()
    .reset_index(name="document_count")
)

duplicate_hash_groups = duplicate_hash_groups[
    duplicate_hash_groups["document_count"] > 1
]

print(
    "Duplicate content groups:",
    len(duplicate_hash_groups)
)

Duplicate content groups: 0


In [15]:
# ======== Detect Suspicious Repeated Lines ==========

def repeated_line_statistics(text):
    lines = [
        line.strip()
        for line in text.splitlines()
        if line.strip()
    ]

    counts = Counter(lines)

    repeated = {
        line: count
        for line, count in counts.items()
        if count > 1
    }

    return {
        "unique_nonempty_lines": len(set(lines)),
        "repeated_line_types": len(repeated),
        "max_line_repetition": max(repeated.values())
        if repeated else 1
    }


line_repeat_data = raw_df["text"].apply(
    repeated_line_statistics
)

line_repeat_df = pd.DataFrame(
    line_repeat_data.tolist()
)

raw_df = pd.concat(
    [raw_df, line_repeat_df],
    axis=1
)

display(
    raw_df[
        [
            "file_name",
            "unique_nonempty_lines",
            "repeated_line_types",
            "max_line_repetition"
        ]
    ].sort_values(
        "repeated_line_types",
        ascending=False
    ).head(20)
)

,file_name,unique_nonempty_lines,repeated_line_types,max_line_repetition
120,Transformer_(deep_learning).txt,679,149,127
53,Graph_neural_network.txt,331,90,47
103,Reinforcement_learning.txt,349,64,58
13,Attention_(machine_learning).txt,196,53,41
97,Q-learning.txt,207,51,66
101,Recurrent_neural_network.txt,312,47,81
105,Residual_neural_network.txt,174,45,63
125,Triangulation_(computer_vision).txt,169,40,51
25,Convolutional_neural_network.txt,340,36,13
57,Homography_(computer_vision).txt,102,34,21


In [16]:
# ======== Overall Inspection Summary =========

inspection_summary = {
    "total_documents": len(raw_df),

    "empty_documents": (
        raw_df["text"].str.strip() == ""
    ).sum(),

    "very_short_documents": (
        raw_df["word_count"] < SHORT_DOCUMENT_THRESHOLD
    ).sum(),

    "documents_with_control_characters": (
        raw_df["control_character_count"] > 0
    ).sum(),

    "documents_with_whitespace_issues": (
        whitespace_issue_mask
    ).sum(),

    "duplicate_content_groups": len(
        duplicate_hash_groups
    ),

    "total_words": raw_df["word_count"].sum(),

    "total_characters": raw_df["character_count"].sum(),

    "minimum_words": raw_df["word_count"].min(),

    "maximum_words": raw_df["word_count"].max(),

    "average_words": raw_df["word_count"].mean(),

    "median_words": raw_df["word_count"].median(),

    "minimum_characters": raw_df["character_count"].min(),

    "maximum_characters": raw_df["character_count"].max(),

    "average_characters": raw_df["character_count"].mean(),

    "median_characters": raw_df["character_count"].median()
}

inspection_summary_df = pd.DataFrame(
    inspection_summary.items(),
    columns=["Metric", "Value"]
)

display(inspection_summary_df)

,Metric,Value
0,total_documents,1.280000e+02
1,empty_documents,0.000000e+00
2,very_short_documents,1.000000e+00
3,documents_with_control_characters,0.000000e+00
4,documents_with_whitespace_issues,7.800000e+01
5,duplicate_content_groups,0.000000e+00
6,total_words,3.578760e+05
7,total_characters,2.583893e+06
8,minimum_words,6.100000e+01
9,maximum_words,2.027900e+04


In [17]:
# ======== Final Raw Document Inspection ========

# Display corpus-level statistics
print("RAW DOCUMENT INSPECTION SUMMARY")
print("=" * 60)

print(f"Total documents              : {len(raw_df):,}")
print(f"Total words                  : {raw_df['word_count'].sum():,}")
print(f"Total characters             : {raw_df['character_count'].sum():,}")
print(f"Empty documents              : {(raw_df['text'].str.strip() == '').sum():,}")
print(
    f"Documents < {SHORT_DOCUMENT_THRESHOLD} words"
    f" : {(raw_df['word_count'] < SHORT_DOCUMENT_THRESHOLD).sum():,}"
)
print(
    f"Whitespace issue documents   : {whitespace_issue_mask.sum():,}"
)
print(
    f"Control-character documents  : "
    f"{(raw_df['control_character_count'] > 0).sum():,}"
)
print(
    f"Duplicate content groups     : "
    f"{len(duplicate_hash_groups):,}"
)

print("\nDOCUMENT LENGTH STATISTICS")
print("=" * 60)

display(
    raw_df[
        [
            "character_count",
            "word_count",
            "line_count",
            "blank_line_count"
        ]
    ].describe()
)

RAW DOCUMENT INSPECTION SUMMARY
Total documents              : 128
Total words                  : 357,876
Total characters             : 2,583,893
Empty documents              : 0
Documents < 100 words : 1
Whitespace issue documents   : 78
Control-character documents  : 0
Duplicate content groups     : 0

DOCUMENT LENGTH STATISTICS


,character_count,word_count,line_count,blank_line_count,blank_line_count
count,128.000000,128.000000,128.000000,128.000000,128.000000
mean,20186.664062,2795.906250,306.632812,155.062500,155.062500
std,23324.972519,3208.514517,637.185802,372.134696,372.134696
min,414.000000,61.000000,10.000000,4.000000,4.000000
25%,4221.250000,602.000000,42.750000,14.000000,14.000000
50%,12635.500000,1633.000000,97.000000,37.500000,37.500000
75%,30147.500000,4068.500000,268.250000,112.500000,112.500000
max,138132.000000,20279.000000,5682.000000,3258.000000,3258.000000


In [18]:
# ======== Inspect Shortest Documents ========

shortest_documents = raw_df.sort_values(
    "word_count",
    ascending=True
)

for _, row in shortest_documents.head(5).iterrows():

    print("=" * 100)
    print(f"FILE       : {row['file_name']}")
    print(f"WORD COUNT : {row['word_count']}")
    print(f"CHARACTERS : {row['character_count']}")
    print("=" * 100)

    print(row["text"][:3000])
    print()

FILE       : ChromaDB.txt
WORD COUNT : 61
CHARACTERS : 414
Chroma or ChromaDB is open-source data infrastructure tailored to applications with large language models.
Its headquarters are in San Francisco. In April 2023, it raised 18 million US dollars as seed funding.
ChromaDB has been used in academic studies on artificial intelligence, particularly as part of the tech stack for retrieval-augmented generation.


== References ==


== External links ==
Official website

FILE       : Feature_store.txt
WORD COUNT : 130
CHARACTERS : 935
A feature store is a centralized repository used in machine learning to store, manage, and serve features for model training and inference. It provides a unified interface for data scientists and engineers to access curated, reusable features derived from raw data, ensuring consistency between training and production environments. Feature stores typically support batch and real-time data pipelines, enabling efficient feature computation, storage, and retri

In [19]:
# ======== Manual Raw Text Inspection ========

sample_file = raw_files[0]

sample_text = sample_file.read_text(
    encoding="utf-8",
    errors="replace"
)

print("FILE:", sample_file.name)
print("=" * 100)
print(sample_text[:5000])

FILE: 1.58-bit_large_language_model.txt
A 1.58-bit large language model (also known as a ternary LLM) is a type of large language model (LLM) designed to be computationally efficient. It achieves this by using weights that are restricted to only three values: −1, 0, and +1. This restriction significantly reduces the model's memory footprint and allows for faster processing, as computationally expensive multiplication operations can be replaced with lower-cost additions. This contrasts with traditional models that use 16-bit floating-point numbers (FP16 or BF16) for their weights.
Studies have shown that for models up to several billion parameters, the performance of 1.58-bit LLMs on various tasks is comparable to their full-precision counterparts. This approach could enable powerful AI to run on less specialized and lower-power hardware.
The name "1.58-bit" comes from the fact that a value with three states contains 
  
    
      
        
          log
          
            2
      

In [20]:
# ============================================================
# Cleaning Configuration
# ============================================================

MAX_CONSECUTIVE_BLANK_LINES = 2

print("Cleaning configuration loaded.")
print(
    "Maximum consecutive blank lines:",
    MAX_CONSECUTIVE_BLANK_LINES
)

Cleaning configuration loaded.
Maximum consecutive blank lines: 2


In [21]:
# ============================================================
# Text Cleaning and Normalization Function
# ============================================================

def clean_text(text):
    """
    Clean and normalize Wikipedia text conservatively.

    The function:
    - Normalizes Unicode
    - Normalizes line endings
    - Removes unsafe control characters
    - Converts tabs to spaces
    - Collapses excessive horizontal whitespace
    - Limits consecutive blank lines
    - Removes leading/trailing whitespace
    """

    # --------------------------------------------------------
    # 1. Unicode normalization
    # --------------------------------------------------------
    text = unicodedata.normalize("NFC", text)

    # --------------------------------------------------------
    # 2. Normalize line endings
    # --------------------------------------------------------
    text = text.replace("\r\n", "\n")
    text = text.replace("\r", "\n")

    # --------------------------------------------------------
    # 3. Remove unsafe control characters
    # --------------------------------------------------------
    cleaned_characters = []

    for char in text:
        category = unicodedata.category(char)

        if category == "Cc" and char not in "\n\t":
            continue

        cleaned_characters.append(char)

    text = "".join(cleaned_characters)

    # --------------------------------------------------------
    # 4. Convert tabs to spaces
    # --------------------------------------------------------
    text = text.replace("\t", " ")

    # --------------------------------------------------------
    # 5. Normalize horizontal whitespace
    # --------------------------------------------------------
    text = re.sub(r"[ ]{2,}", " ", text)

    # --------------------------------------------------------
    # 6. Remove spaces around line boundaries
    # --------------------------------------------------------
    lines = [
        line.strip()
        for line in text.split("\n")
    ]

    # --------------------------------------------------------
    # 7. Limit consecutive blank lines
    # --------------------------------------------------------
    normalized_lines = []

    blank_line_count = 0

    for line in lines:

        if line == "":
            blank_line_count += 1

            if blank_line_count <= MAX_CONSECUTIVE_BLANK_LINES:
                normalized_lines.append(line)

        else:
            blank_line_count = 0
            normalized_lines.append(line)

    # --------------------------------------------------------
    # 8. Reconstruct document
    # --------------------------------------------------------
    text = "\n".join(normalized_lines)

    # --------------------------------------------------------
    # 9. Final trim
    # --------------------------------------------------------
    text = text.strip()

    return text

In [22]:
# ============================================================
# Test Cleaning Function
# ============================================================

sample_text = raw_df.loc[0, "text"]

cleaned_sample = clean_text(sample_text)

print("Original characters :", len(sample_text))
print("Cleaned characters  :", len(cleaned_sample))

print("\nOriginal preview:")
print(sample_text[:1000])

print("\n" + "=" * 100)

print("\nCleaned preview:")
print(cleaned_sample[:1000])

Original characters : 4395
Cleaned characters  : 4272

Original preview:
A 1.58-bit large language model (also known as a ternary LLM) is a type of large language model (LLM) designed to be computationally efficient. It achieves this by using weights that are restricted to only three values: −1, 0, and +1. This restriction significantly reduces the model's memory footprint and allows for faster processing, as computationally expensive multiplication operations can be replaced with lower-cost additions. This contrasts with traditional models that use 16-bit floating-point numbers (FP16 or BF16) for their weights.
Studies have shown that for models up to several billion parameters, the performance of 1.58-bit LLMs on various tasks is comparable to their full-precision counterparts. This approach could enable powerful AI to run on less specialized and lower-power hardware.
The name "1.58-bit" comes from the fact that a value with three states contains 
  
    
      
        
          lo

In [23]:
# ============================================================
# Sample Before/After Comparison
# ============================================================

original_words = len(
    re.findall(r"\S+", sample_text)
)

cleaned_words = len(
    re.findall(r"\S+", cleaned_sample)
)

print("Original word count:", original_words)
print("Cleaned word count :", cleaned_words)
print(
    "Word-count difference:",
    cleaned_words - original_words
)


# ============================================================
# Unicode Preservation Test
# ============================================================

unicode_test = """
Machine learning uses β and α parameters.
The result is approximately → 1.58.
The temperature is 20 °C.
"""

unicode_cleaned = clean_text(unicode_test)

print(unicode_cleaned)

Original word count: 609
Cleaned word count : 609
Word-count difference: 0
Machine learning uses β and α parameters.
The result is approximately → 1.58.
The temperature is 20 °C.


In [24]:
# ============================================================
# Control Character Test
# ============================================================

control_test = (
    "Wikipedia article\twith formatting"
    "\x00"
    "and unsafe control character."
)

control_cleaned = clean_text(control_test)

print(repr(control_cleaned))

'Wikipedia article with formattingand unsafe control character.'


In [25]:
# ============================================================
# Cleaning Function Assertions
# ============================================================

test_text = (
    "  This   is a test.\n\n\n\n"
    "Second\tline with β.\n"
    "Third line.\x00  "
)

test_cleaned = clean_text(test_text)

assert "\x00" not in test_cleaned
assert "\t" not in test_cleaned
assert "  " not in test_cleaned
assert "β" in test_cleaned
assert "This is a test." in test_cleaned
assert test_cleaned.endswith("Third line.")

print("All cleaning-function tests passed.")

All cleaning-function tests passed.


In [26]:
# ============================================================
# STEP 3 – APPLY CLEANING TO ALL RAW DOCUMENTS
# ============================================================

print("Raw directory     :", RAW_DIR)
print("Cleaned directory :", CLEANED_DIR)

raw_files = sorted(RAW_DIR.glob("*.txt"))
existing_cleaned_files = sorted(CLEANED_DIR.glob("*.txt"))

print("\nRaw TXT files              :", len(raw_files))
print("Existing cleaned TXT files:", len(existing_cleaned_files))

if len(raw_files) != 128:
    raise ValueError(
        f"Expected 128 raw documents, found {len(raw_files)}."
    )

print("\nRaw document count validation passed.")

Raw directory     : c:\Users\RAaM\OneDrive\Desktop\Dynamic AI Knowledge Assistant\data\raw\wikipedia
Cleaned directory : c:\Users\RAaM\OneDrive\Desktop\Dynamic AI Knowledge Assistant\data\processed\cleaned

Raw TXT files              : 128
Existing cleaned TXT files: 128

Raw document count validation passed.


In [27]:
# ============================================================
# Clean and save all Wikipedia documents
# ============================================================

cleaning_results = []

for raw_file in sorted(RAW_DIR.glob("*.txt")):

    # Read raw document
    original_text = raw_file.read_text(
        encoding="utf-8",
        errors="replace"
    )

    # Apply validated cleaning function
    cleaned_text = clean_text(original_text)

    # Destination file
    cleaned_file = CLEANED_DIR / raw_file.name

    # Save cleaned document
    cleaned_file.write_text(
        cleaned_text,
        encoding="utf-8"
    )

    # Collect statistics
    original_word_count = len(
        re.findall(r"\S+", original_text)
    )

    cleaned_word_count = len(
        re.findall(r"\S+", cleaned_text)
    )

    cleaning_results.append({
        "file_name": raw_file.name,
        "original_characters": len(original_text),
        "cleaned_characters": len(cleaned_text),
        "original_words": original_word_count,
        "cleaned_words": cleaned_word_count,
        "character_difference": (
            len(cleaned_text) - len(original_text)
        ),
        "word_difference": (
            cleaned_word_count - original_word_count
        )
    })

cleaning_results_df = pd.DataFrame(cleaning_results)

print(
    f"Successfully cleaned and saved "
    f"{len(cleaning_results_df)} documents."
)

Successfully cleaned and saved 128 documents.


In [28]:
# ============================================================
# Verify cleaned document count
# ============================================================

cleaned_files = sorted(
    CLEANED_DIR.glob("*.txt")
)

print("Cleaned TXT files:", len(cleaned_files))

if len(cleaned_files) != len(raw_files):
    raise ValueError(
        "Cleaned document count does not match raw document count."
    )

print("Document count validation passed.")

Cleaned TXT files: 128
Document count validation passed.


In [29]:
# ============================================================
# Verify filenames are preserved
# ============================================================

raw_names = {
    file.name
    for file in raw_files
}

cleaned_names = {
    file.name
    for file in cleaned_files
}

missing_cleaned = raw_names - cleaned_names
unexpected_cleaned = cleaned_names - raw_names

print("Missing cleaned files   :", len(missing_cleaned))
print("Unexpected cleaned files:", len(unexpected_cleaned))

if missing_cleaned:
    print("\nMissing:")
    for name in sorted(missing_cleaned):
        print(name)

if unexpected_cleaned:
    print("\nUnexpected:")
    for name in sorted(unexpected_cleaned):
        print(name)

if not missing_cleaned and not unexpected_cleaned:
    print("\nFilename validation passed.")

Missing cleaned files   : 0
Unexpected cleaned files: 0

Filename validation passed.


In [30]:
# ============================================================
# Before/After Cleaning Statistics
# ============================================================

display(
    cleaning_results_df[
        [
            "original_characters",
            "cleaned_characters",
            "original_words",
            "cleaned_words",
            "character_difference",
            "word_difference"
        ]
    ].describe()
)

,original_characters,cleaned_characters,original_words,cleaned_words,character_difference,word_difference
count,128.000000,128.000000,128.000000,128.000000,128.000000,128.0
mean,20186.664062,18549.281250,2795.906250,2795.906250,-1637.382812,0.0
std,23324.972519,21367.923484,3208.514517,3208.514517,5831.338771,0.0
min,414.000000,414.000000,61.000000,61.000000,-54669.000000,0.0
25%,4221.250000,4193.500000,602.000000,602.000000,-34.750000,0.0
50%,12635.500000,11288.000000,1633.000000,1633.000000,-2.500000,0.0
75%,30147.500000,26291.750000,4068.500000,4068.500000,0.000000,0.0
max,138132.000000,138120.000000,20279.000000,20279.000000,0.000000,0.0


In [31]:
# ============================================================
# Inspect word-count changes
# ============================================================

word_changes = cleaning_results_df[
    cleaning_results_df["word_difference"] != 0
].copy()

print(
    "Documents with word-count changes:",
    len(word_changes)
)

display(
    word_changes.sort_values(
        "word_difference"
    )
)

Documents with word-count changes: 0


,file_name,original_characters,cleaned_characters,original_words,cleaned_words,character_difference,word_difference


In [32]:
# ============================================================
# Inspect character-count changes
# ============================================================

character_changes = (
    cleaning_results_df
    .sort_values(
        "character_difference"
    )
)

display(
    character_changes[
        [
            "file_name",
            "original_characters",
            "cleaned_characters",
            "character_difference"
        ]
    ].head(20)
)

,file_name,original_characters,cleaned_characters,character_difference
120,Transformer_(deep_learning).txt,122367,67698,-54669
53,Graph_neural_network.txt,44701,26978,-17723
101,Recurrent_neural_network.txt,54902,38203,-16699
125,Triangulation_(computer_vision).txt,31713,15346,-16367
105,Residual_neural_network.txt,25797,13524,-12273
97,Q-learning.txt,30641,18547,-12094
103,Reinforcement_learning.txt,52531,40532,-11999
13,Attention_(machine_learning).txt,25669,15475,-10194
41,Feature_scaling.txt,14555,6163,-8392
57,Homography_(computer_vision).txt,12684,5904,-6780


In [33]:
# ============================================================
# Verify legitimate short document preservation
# ============================================================

chromadb_file = CLEANED_DIR / "ChromaDB.txt"

if not chromadb_file.exists():
    raise FileNotFoundError(
        "ChromaDB.txt was not found in cleaned directory."
    )

chromadb_text = chromadb_file.read_text(
    encoding="utf-8"
)

chromadb_words = len(
    re.findall(r"\S+", chromadb_text)
)

print("ChromaDB.txt cleaned word count:", chromadb_words)

if not chromadb_text.strip():
    raise ValueError(
        "ChromaDB.txt became empty after cleaning."
    )

print("ChromaDB.txt preservation check passed.")

ChromaDB.txt cleaned word count: 61
ChromaDB.txt preservation check passed.


In [34]:
# ============================================================
# Verify raw documents remain unchanged
# ============================================================

raw_hash_validation = []

for raw_file in sorted(RAW_DIR.glob("*.txt")):

    current_text = raw_file.read_text(
        encoding="utf-8",
        errors="replace"
    )

    current_hash = content_hash(current_text)

    raw_hash_validation.append({
        "file_name": raw_file.name,
        "current_hash": current_hash
    })

current_raw_hashes_df = pd.DataFrame(
    raw_hash_validation
)

original_raw_hashes_df = raw_df[
    [
        "file_name",
        "content_hash"
    ]
].copy()

raw_hash_comparison = original_raw_hashes_df.merge(
    current_raw_hashes_df,
    on="file_name",
    how="outer"
)

raw_hash_comparison["unchanged"] = (
    raw_hash_comparison["content_hash"]
    ==
    raw_hash_comparison["current_hash"]
)

print(
    "Raw documents unchanged:",
    raw_hash_comparison["unchanged"].all()
)

if not raw_hash_comparison["unchanged"].all():
    raise ValueError(
        "WARNING: One or more raw documents changed!"
    )

print("Raw-file integrity validation passed.")

Raw documents unchanged: True
Raw-file integrity validation passed.


In [35]:
# ============================================================
# Raw vs Cleaned Sample
# ============================================================

sample_name = raw_files[0].name

raw_sample = (
    RAW_DIR / sample_name
).read_text(
    encoding="utf-8",
    errors="replace"
)

cleaned_sample = (
    CLEANED_DIR / sample_name
).read_text(
    encoding="utf-8"
)

print("FILE:", sample_name)

print("\n" + "=" * 100)
print("RAW TEXT")
print("=" * 100)
print(raw_sample[:3000])

print("\n" + "=" * 100)
print("CLEANED TEXT")
print("=" * 100)
print(cleaned_sample[:3000])

FILE: 1.58-bit_large_language_model.txt

RAW TEXT
A 1.58-bit large language model (also known as a ternary LLM) is a type of large language model (LLM) designed to be computationally efficient. It achieves this by using weights that are restricted to only three values: −1, 0, and +1. This restriction significantly reduces the model's memory footprint and allows for faster processing, as computationally expensive multiplication operations can be replaced with lower-cost additions. This contrasts with traditional models that use 16-bit floating-point numbers (FP16 or BF16) for their weights.
Studies have shown that for models up to several billion parameters, the performance of 1.58-bit LLMs on various tasks is comparable to their full-precision counterparts. This approach could enable powerful AI to run on less specialized and lower-power hardware.
The name "1.58-bit" comes from the fact that a value with three states contains 
  
    
      
        
          log
          
          

In [36]:
# ============================================================
# COMPLETE CLEANED DATASET VALIDATION
# ============================================================

# ------------------------------------------------------------
# 1. Document count
# ------------------------------------------------------------

raw_files = sorted(RAW_DIR.glob("*.txt"))
cleaned_files = sorted(CLEANED_DIR.glob("*.txt"))

assert len(raw_files) == 128
assert len(cleaned_files) == 128

print("Document count:")
print("  Raw     :", len(raw_files))
print("  Cleaned :", len(cleaned_files))


# ------------------------------------------------------------
# 2. Filename validation
# ------------------------------------------------------------

raw_names = {file.name for file in raw_files}
cleaned_names = {file.name for file in cleaned_files}

assert raw_names == cleaned_names

print("\nFilename validation: PASSED")


# ------------------------------------------------------------
# 3. Read cleaned documents
# ------------------------------------------------------------

cleaned_documents = []

for file_path in cleaned_files:

    text = file_path.read_text(
        encoding="utf-8"
    )

    cleaned_documents.append({
        "file_name": file_path.name,
        "text": text,
        "character_count": len(text),
        "word_count": len(
            re.findall(r"\S+", text)
        )
    })

cleaned_df = pd.DataFrame(cleaned_documents)


# ------------------------------------------------------------
# 4. Empty-document validation
# ------------------------------------------------------------

empty_cleaned = cleaned_df[
    cleaned_df["text"].str.strip() == ""
]

print("\nEmpty cleaned documents:", len(empty_cleaned))

assert len(empty_cleaned) == 0


# ============================================================
# 5. WORD-COUNT PRESERVATION VALIDATION
# ============================================================

word_changes = cleaning_results_df[
    cleaning_results_df["word_difference"] != 0
].copy()

print(
    "Documents with word-count changes:",
    len(word_changes)
)

if len(word_changes) > 0:
    display(
        word_changes[
            [
                "file_name",
                "original_words",
                "cleaned_words",
                "word_difference"
            ]
        ]
    )

    raise ValueError(
        "Unexpected word-count changes detected."
    )

print("Word-count preservation validation passed.")

# ------------------------------------------------------------
# 6. Control-character validation
# ------------------------------------------------------------

def find_unsafe_controls(text):
    return [
        char
        for char in text
        if unicodedata.category(char) == "Cc"
        and char not in "\n"
    ]


control_results = []

for _, row in cleaned_df.iterrows():

    controls = find_unsafe_controls(
        row["text"]
    )

    control_results.append({
        "file_name": row["file_name"],
        "unsafe_control_count": len(controls)
    })

control_df = pd.DataFrame(control_results)

unsafe_control_documents = control_df[
    control_df["unsafe_control_count"] > 0
]

print(
    "Documents with unsafe control characters:",
    len(unsafe_control_documents)
)

assert len(unsafe_control_documents) == 0


# ------------------------------------------------------------
# 7. Tab validation
# ------------------------------------------------------------

tab_documents = cleaned_df[
    cleaned_df["text"].str.contains(
        "\t",
        regex=False
    )
]

print(
    "Documents containing tabs:",
    len(tab_documents)
)

assert len(tab_documents) == 0


# ------------------------------------------------------------
# 8. Legitimate short-document validation
# ------------------------------------------------------------

chromadb_text = (
    CLEANED_DIR / "ChromaDB.txt"
).read_text(
    encoding="utf-8"
)

chromadb_words = len(
    re.findall(r"\S+", chromadb_text)
)

print(
    "ChromaDB.txt word count:",
    chromadb_words
)

assert chromadb_words == 61


# ------------------------------------------------------------
# 9. Raw-file integrity
# ------------------------------------------------------------

raw_integrity = []

for raw_file in raw_files:

    current_text = raw_file.read_text(
        encoding="utf-8",
        errors="replace"
    )

    current_hash = content_hash(
        current_text
    )

    original_hash = raw_df.loc[
        raw_df["file_name"] == raw_file.name,
        "content_hash"
    ].iloc[0]

    raw_integrity.append(
        current_hash == original_hash
    )

assert all(raw_integrity)

print(
    "Raw-file integrity: PASSED"
)


# ------------------------------------------------------------
# 10. Final statistics
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("PHASE 2 – CLEANED DATASET VALIDATION")
print("=" * 70)

print(
    f"Documents                 : {len(cleaned_df):,}"
)

print(
    f"Total words               : "
    f"{cleaned_df['word_count'].sum():,}"
)

print(
    f"Total characters          : "
    f"{cleaned_df['character_count'].sum():,}"
)

print(
    f"Empty documents           : "
    f"{len(empty_cleaned):,}"
)

print(
    f"Word-count changes        : "
    f"{len(word_changes):,}"
)

print(
    f"Unsafe-control documents  : "
    f"{len(unsafe_control_documents):,}"
)

print(
    f"Documents containing tabs : "
    f"{len(tab_documents):,}"
)

print(
    f"Duplicate raw documents   : "
    f"{len(duplicate_hash_groups):,}"
)

print("=" * 70)

Document count:
  Raw     : 128
  Cleaned : 128

Filename validation: PASSED

Empty cleaned documents: 0
Documents with word-count changes: 0
Word-count preservation validation passed.
Documents with unsafe control characters: 0
Documents containing tabs: 0
ChromaDB.txt word count: 61
Raw-file integrity: PASSED

PHASE 2 – CLEANED DATASET VALIDATION
Documents                 : 128
Total words               : 357,876
Total characters          : 2,374,308
Empty documents           : 0
Word-count changes        : 0
Unsafe-control documents  : 0
Documents containing tabs : 0
Duplicate raw documents   : 0


In [37]:
# ============================================================
# PHASE 2 FINAL PASS / FAIL
# ============================================================

phase2_passed = (
    len(raw_files) == 128
    and len(cleaned_files) == 128
    and len(empty_cleaned) == 0
    and len(word_changes) == 0
    and len(unsafe_control_documents) == 0
    and len(tab_documents) == 0
    and chromadb_words == 61
    and all(raw_integrity)
    and len(duplicate_hash_groups) == 0
)

print("=" * 70)

if phase2_passed:
    print("PHASE 2 VALIDATION PASSED")
else:
    print("PHASE 2 VALIDATION FAILED")

print("=" * 70)

PHASE 2 VALIDATION PASSED
